# Matrix-free linear algebra

This guide demonstrates SpaceCore's iterative linear algebra routines on `MatrixFreeLinOp` objects.

A matrix-free operator is useful when the action of an operator is cheap, but building or storing the full matrix would be wasteful. In SpaceCore, the only requirements are:

- a domain space,
- a codomain space,
- a forward action `apply(x)`,
- an adjoint action `rapply(y)`.

The solvers below never need a dense matrix representation of the operator.

In [50]:
import numpy as np

import spacecore as sc

## Backend context and vector space

We use NumPy here to keep the notebook easy to run. The same operators can be converted to other supported backends when their callbacks are written using backend-compatible operations.

In [51]:
ctx = sc.Context(sc.NumpyOps(), dtype=np.float64, enable_checks=False)
n = 50000
X = sc.VectorSpace((n,), ctx)

grid = np.linspace(0.0, np.pi, n)

## A square Hermitian positive-definite operator

This operator acts like a positive diagonal matrix, but we do not build a matrix object. The callback stores only the diagonal coefficients and multiplies elementwise.

Because the operator is real and self-adjoint, the forward and adjoint callbacks are the same function.

In [52]:
diag = ctx.asarray(np.concatenate(([0.25], np.linspace(1.0, 2.0, n - 2), [6.0])))

def spd_apply(x):
    return diag * x

A = sc.MatrixFreeLinOp(spd_apply, spd_apply, X, X, ctx)

A.domain.shape, A.codomain.shape

((50000,), (50000,))

## Conjugate gradient: solve `A x = b`

`cg` is for square Hermitian positive-definite systems. We create a known solution, apply the operator to get the right-hand side, and ask CG to recover the solution.

The result object summarizes convergence metadata and avoids printing the full solution vector in its `repr`.

In [53]:
x_true = ctx.asarray(np.sin(grid) + 0.2 * np.cos(3.0 * grid))
b = A.apply(x_true)

cg_result = sc.cg(A, b, tol=1e-8, maxiter=256)
cg_result

CGResult(converged=True, num_iters=64, residual_norm=1.18582e-08, x=<array shape=(50000,), dtype=float64>)

In [54]:
relative_error = ctx.ops.norm(cg_result.x - x_true) / ctx.ops.norm(x_true)
float(relative_error)

4.5131997046538686e-11

## A rectangular matrix-free operator

`lsqr` works for rectangular least-squares problems. Here `B : R^n -> R^{2n}` maps a vector into two measurement channels:

$$
B x = \begin{bmatrix}x \\ w \odot x\end{bmatrix}.
$$

The adjoint combines the two channels:

$$
B^* y = y_1 + w \odot y_2.
$$

Again, no matrix is built.

In [55]:
Y = sc.VectorSpace((2 * n,), ctx)
weights = ctx.asarray(np.linspace(0.5, 1.5, n))

def rectangular_apply(x):
    return ctx.ops.concatenate((x, weights * x), axis=0)

def rectangular_rapply(y):
    return y[:n] + weights * y[n:]

B = sc.MatrixFreeLinOp(rectangular_apply, rectangular_rapply, X, Y, ctx)

B.domain.shape, B.codomain.shape

((50000,), (100000,))

## LSQR: solve a least-squares problem

We add a small deterministic perturbation to the measurements so the problem is not just a perfectly consistent copy of the original vector. LSQR minimizes `||B x - data||` using `B.apply` and `B.H.apply` internally.

In [56]:
x_ls_true = ctx.asarray(np.cos(2.0 * grid))
noise = 0.01 * ctx.asarray(np.sin(np.linspace(0.0, 4.0 * np.pi, 2 * n)))
data = B.apply(x_ls_true) + noise

lsqr_result = sc.lsqr(B, data, tol=1e-10, maxiter=256)
lsqr_result

LSQRResult(converged=True, num_iters=64, residual_norm=0.304159, normal_residual_norm=8.83974e-14, x=<array shape=(50000,), dtype=float64>)

In [57]:
normal_residual = ctx.ops.norm(B.H.apply(B.apply(lsqr_result.x) - data))
float(normal_residual)

8.839736157139945e-14

## Power iteration: estimate the dominant eigenpair

`power_iteration` estimates the largest-magnitude eigenvalue of a square operator. For our diagonal example, the answer should be the largest entry in `diag`.

In [58]:
x0 = ctx.asarray(np.ones(n))
power_result = sc.power_iteration(A, x0=x0, tol=1e-10, maxiter=256)
power_result

PowerIterationResult(converged=True, num_iters=64, eigenvalue=6, residual_norm=3.25687e-29, eigenvector=<array shape=(50000,), dtype=float64>)

In [59]:
float(power_result.eigenvalue), float(diag[-1])

(6.0, 6.0)

## Stochastic Lanczos: estimate the smallest eigenpair

`stochastic_lanczos` builds a Krylov subspace from an initial domain element and returns a Ritz approximation to the smallest eigenpair. The returned eigenvector is a member of `A.domain`, not a raw matrix column from an internal representation.

In [60]:
initial = ctx.asarray(np.ones(n))
smallest_eigenvalue, smallest_eigenvector = sc.stochastic_lanczos(
    A,
    initial,
    max_iter=64,
    tol=1e-10,
)

float(smallest_eigenvalue), smallest_eigenvector.shape

(0.25, (50000,))

In [61]:
ritz_residual = ctx.ops.norm(A.apply(smallest_eigenvector) - smallest_eigenvalue * smallest_eigenvector)
float(ritz_residual), float(diag[0])

(6.179265594934942e-15, 0.25)

## What to take away

- `MatrixFreeLinOp` lets you use iterative algorithms without building a matrix.
- `cg` is for square Hermitian positive-definite solves.
- `lsqr` is for general rectangular least-squares problems.
- `power_iteration` gives a dominant eigenpair estimate.
- `stochastic_lanczos` gives a smallest Ritz eigenpair estimate from a Krylov subspace.
- Solver result objects are compact to display, while full arrays remain available as attributes such as `cg_result.x`.